# Fase 7 — Fine-Tuning: de modelo base a asistente 🎛️

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/REPO/blob/main/notebooks/07_finetuning.ipynb)

> De un modelo base que solo *continúa texto* a un asistente que *sigue instrucciones*.
> Implementamos **LoRA desde cero**, SFT con masking del prompt, chat templates,
> y derivamos **QLoRA**, **RLHF** y **DPO** con verificaciones numéricas.

## Objetivos de aprendizaje
- Entender la diferencia base ↔ instruct y las 3 etapas del alineamiento.
- Implementar **LoRA desde cero** ($\Delta W = \frac{\alpha}{r} B A$) sobre un MiniGPT toy,
  entrenar con curva de loss y comparar geometría de $A$, $B$, $\Delta W$.
- Construir un dataset SFT con 50 + ejemplos multi-dominio y analizar estadísticas.
- Derivar matemáticamente **DPO** y verificarla numéricamente en múltiples escenarios.
- Guía práctica de hiperparámetros LoRA ($r$, $\alpha$, `target_modules`, LR).

## Prerrequisitos
Fases 5–6. Tiempo estimado: **SMALL** ~40 min · **LARGE** ~70 min.
Todos los modelos HF usan `try/except` + fallback; el núcleo corre en CPU.

## Índice
1. Base vs Instruct · 2. Coste del FT completo · 3. Dataset SFT propio ·
4. Chat templates · 5. SFT con masking · 6. LoRA desde cero ·
7. LoRA con PEFT · 8. QLoRA · 9. Before/After · 10. RLHF ·
11. DPO · 11b. Hiperparámetros LoRA · 12. Resumen

## Setup — dependencias, hardware, semillas

In [ ]:
import sys, subprocess, math, json, random, collections, textwrap
import os

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# Torch
try:
    import torch
except ImportError:
    pipq("torch")
    import torch

import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted")
except ImportError:
    pipq("seaborn"); import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted")

# Colab?
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Hardware
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                       "mps"  if torch.backends.mps.is_available() else "cpu")

# Seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

plt.rcParams["figure.figsize"] = (10, 4)

# Info GPU
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB")
elif DEVICE.type == "mps":
    vram_gb = 0; print("Apple Silicon MPS")
else:
    vram_gb = 0; print("CPU only")

print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | In Colab: {IN_COLAB}")

## Configuración SMALL / LARGE

| Parámetro | SMALL (T4 / CPU) | LARGE (A100) |
|-----------|-------------------|--------------|
| Modelo base | SmolLM2-135M | Qwen2.5-1.5B |
| LoRA r | 8 | 16 |
| LoRA alpha | 16 | 32 |
| LR | 2e-4 | 1e-4 |
| QLoRA | No | Sí |

In [ ]:
# Auto-detección de perfil (o forzar manualmente: PROFILE = "small")
PROFILE = "auto"
if PROFILE == "auto":
    PROFILE = "large" if vram_gb >= 24 else "small"

CONFIG = {
    "small": {
        "base_model":    "HuggingFaceTB/SmolLM2-135M",
        "fallback_model":"sshleifer/tiny-gpt2",
        "lora_r":        8,
        "lora_alpha":    16,
        "lora_dropout":  0.05,
        "lr":            2e-4,
        "use_qlora":     False,
        "batch_size":    32,
        "max_steps":     3000,
        "warmup_steps":  200,
    },
    "large": {
        "base_model":    "Qwen/Qwen2.5-1.5B",
        "fallback_model":"sshleifer/tiny-gpt2",
        "lora_r":        16,
        "lora_alpha":    32,
        "lora_dropout":  0.05,
        "lr":            1e-4,
        "use_qlora":     True,
        "batch_size":    64,
        "max_steps":     20000,
        "warmup_steps":  1000,
    },
}
C = CONFIG[PROFILE]
print(f"PROFILE = {PROFILE}")
print(f"Modelo base sugerido : {C['base_model']}")
print(f"LoRA r={C['lora_r']}  alpha={C['lora_alpha']}  lr={C['lr']}")

## 1. Base vs Instruct: las 3 etapas del alineamiento

Un modelo **base** (pre-entrenado con next-token prediction) solo sabe *continuar* texto
plausible.  No "responde": ante "¿Qué es un transformador?" podría continuar con
"¿Qué es un capacitor? ¿Qué es un inductor?…"

Para construir un **asistente** se necesita post-entrenamiento en tres etapas:

```mermaid
flowchart LR
    PT["Pre-training<br/>(next-token, Fase 6)"] --> SFT["SFT<br/>(instruction tuning)"]
    SFT --> RM["Reward model<br/>(preferencias humanas)"]
    RM --> RLHF["RLHF / DPO<br/>(alineamiento)"]
```

| Etapa | Qué aprende | Datos |
|-------|------------|-------|
| **SFT** | Seguir el formato instrucción→respuesta | Pares I/O de calidad |
| **RM** | Qué respuesta prefieren los humanos | Comparaciones A vs B |
| **RLHF / DPO** | Maximizar reward humano dentro de una restricción KL | Pares (preferida, rechazada) |

> 📌 **LIMA** (Zhou 2023): 1 000 ejemplos *excelentes* superan a 50 000 mediocres.
> La calidad del dataset SFT importa más que la cantidad.

In [ ]:
# Diferencia cualitativa: ejemplo de cómo un base continúa vs un instruct responde
base_continuation = (
    "¿Qué es el flujo de carga?\n"
    "¿Qué es la regulación de voltaje?\n"
    "¿Qué es la potencia reactiva? ..."
)
instruct_response = (
    "El flujo de carga calcula voltajes nodales y flujos de potencia "
    "en estado estacionario mediante Newton-Raphson o Gauss-Seidel."
)
print("=== Modelo BASE (continúa) ===")
print(textwrap.fill(base_continuation, 70))
print("\n=== Modelo INSTRUCT (responde) ===")
print(textwrap.fill(instruct_response, 70))

## 2. Fine-tuning completo y su coste

Con **AdamW** en fp16/fp32 mixto, cada parámetro requiere:

| Elemento | Bytes |
|----------|-------|
| Pesos fp16 (inferencia) | 2 |
| Pesos fp32 (master copy) | 4 |
| Gradientes fp32 | 4 |
| Momento 1 (Adam) fp32 | 4 |
| Momento 2 (Adam) fp32 | 4 |
| **Total** | **18 bytes/param** (aprox. 16 B en la práctica) |

$$\text{VRAM}_{\text{FT}} \approx 16 \cdot N_{\text{params}} \text{ bytes}$$

Esto hace inviable el FT completo de modelos grandes en una sola GPU → motivación de
**LoRA** (solo entrena $r(d+k)$ params) y **QLoRA** (base en 4-bit).

**Catastrophic forgetting**: FT completo sobre pocos datos puede sobreescribir
conocimiento del pre-entrenamiento.  LoRA lo mitiga porque congela el modelo base.

In [ ]:
models = [
    ("GPT-2 small",   117e6),
    ("SmolLM2-135M",  135e6),
    ("Qwen2.5-1.5B",  1.5e9),
    ("LLaMA-3  8B",   8e9),
    ("LLaMA-3 70B",   70e9),
]

def mem_ft_full(p): return p * 16 / 1e9
def mem_lora(p, r=8, d=4096): return p * 2/1e9 + (r*(d+d)*2*12)/1e9  # base fp16 + adapts

print(f"{'Modelo':<20} {'FT completo':>14} {'LoRA r=8 (aprox)':>18}")
print("-" * 55)
for name, p in models:
    print(f"{name:<20} {mem_ft_full(p):>12.1f} GB  {mem_lora(p):>14.1f} GB")

# Visualización
names  = [n for n, _ in models]
full   = [mem_ft_full(p) for _, p in models]
lora_v = [mem_lora(p)    for _, p in models]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, full,   0.4, label="FT completo (fp16+AdamW)",  color="#e07070")
ax.bar(x + 0.2, lora_v, 0.4, label="LoRA r=8 (base fp16)",      color="#70b0e0")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha="right")
ax.set_ylabel("VRAM estimada (GB)")
ax.set_title("VRAM requerida: FT completo vs LoRA")
ax.axhline(16, color="gray", ls="--", lw=0.8, label="16 GB (T4)")
ax.axhline(40, color="black", ls="--", lw=0.8, label="40 GB (A100)")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Dataset SFT instruccional propio

Construimos un dataset multi-dominio de alta calidad en formato `messages` (ChatML).
Los 21 dominios cubren el stack técnico del laboratorio:

> sistemas de potencia · machine learning · Python · SQL/PostgreSQL · AWS ·
> matemáticas · álgebra lineal · probabilidad · redes · Git · Docker ·
> optimización · series de tiempo · NLP · deep learning · transformers ·
> MLOps · estadística · señales · control · electrónica de potencia

In [ ]:
# ── Dataset SFT: 50+ ejemplos multi-dominio de calidad ──────────────────

SFT_RAW = [
  # ── Sistemas de potencia ──────────────────────────────────────────────
  {"domain":"sistemas_potencia","messages":[
    {"role":"user","content":"¿Qué es el flujo de carga (load flow) en sistemas de potencia?"},
    {"role":"assistant","content":
     "El flujo de carga calcula los voltajes nodales (magnitud y ángulo) y los flujos de potencia activa y reactiva en un sistema eléctrico en estado estacionario. Se resuelven las ecuaciones de balance de potencia P+jQ = V·I* en cada nodo usando métodos iterativos: Newton-Raphson (convergencia cuadrática, preferido en redes grandes) o Gauss-Seidel (simple pero lento). Se clasifica cada nodo como PQ (carga), PV (generador) o slack (referencia angular)."}
  ]},
  {"domain":"sistemas_potencia","messages":[
    {"role":"user","content":"Explica el criterio de seguridad N-1 en redes eléctricas."},
    {"role":"assistant","content":
     "El criterio N-1 exige que el sistema opere dentro de límites normales aunque falle cualquier elemento individual (línea, transformador, generador). Se evalúa simulando la desconexión de cada elemento y verificando que no haya sobrecargas ni violaciones de voltaje. Es el estándar de planificación en Europa (ENTSO-E) y América del Norte (NERC/NERC-FAC-001)."}
  ]},
  {"domain":"sistemas_potencia","messages":[
    {"role":"user","content":"¿Qué diferencia hay entre potencia activa, reactiva y aparente?"},
    {"role":"assistant","content":
     "Potencia activa P [W]: trabajo útil realizado por la corriente. Potencia reactiva Q [VAR]: intercambio de energía entre campo eléctrico/magnético, no hace trabajo útil pero carga los conductores. Potencia aparente S [VA]: S = P + jQ, con |S| = V·I. El factor de potencia fp = P/|S| = cos(φ). Minimizar Q reduce pérdidas I²R; los condensadores se usan como compensadores locales de Q."}
  ]},
  {"domain":"sistemas_potencia","messages":[
    {"role":"user","content":"¿Cómo funciona la regulación de voltaje con un transformador de toma variable (tap changer)?"},
    {"role":"assistant","content":
     "Un OLTC (On-Load Tap Changer) ajusta la relación de transformación n en pasos discretos (±10–15%) bajo carga. Al subir la toma del lado secundario se eleva V₂; al bajarla se reduce. Los reguladores automáticos miden V en el punto de control y activan el motor del tap para mantener el voltaje dentro de ±1–2% del setpoint, con retardo para evitar oscilaciones."}
  ]},
  # ── Machine Learning ──────────────────────────────────────────────────
  {"domain":"machine_learning","messages":[
    {"role":"user","content":"¿Cuál es la diferencia entre bias y varianza en un modelo ML?"},
    {"role":"assistant","content":
     "Bias: error sistemático por suposiciones del modelo (subajuste). Varianza: sensibilidad a fluctuaciones del set de entrenamiento (sobreajuste). Error total ≈ Bias² + Varianza + Ruido irreducible. Un modelo simple (ej. regresión lineal) tiene alto bias y baja varianza; un árbol profundo tiene bajo bias y alta varianza. Regularización, ensembles (bagging) y más datos reducen varianza; añadir features o aumentar capacidad reduce bias."}
  ]},
  {"domain":"machine_learning","messages":[
    {"role":"user","content":"Explica el algoritmo de backpropagation paso a paso."},
    {"role":"assistant","content":
     "1) Forward pass: calcular predicción ŷ y pérdida L. 2) dL/dŷ: gradiente de la pérdida respecto a la salida. 3) Regla de la cadena capa a capa hacia atrás: dL/dW_l = dL/dz_l · dz_l/dW_l, donde z_l = W_l·a_{l-1} + b_l y a_l = σ(z_l). 4) Actualizar pesos: W ← W - η·dL/dW. 5) Repetir para cada lote. La clave es reutilizar los gradientes ya calculados (memoización automática en autograd)."}
  ]},
  {"domain":"machine_learning","messages":[
    {"role":"user","content":"¿Qué es el descenso de gradiente estocástico (SGD) y por qué usar mini-batches?"},
    {"role":"assistant","content":
     "SGD actualiza θ ← θ - η·∇L(θ; x_i) con un solo ejemplo por paso: ruidoso pero escala a datasets grandes. Mini-batch SGD usa B ejemplos (típico 32–512): reduce varianza del gradiente vs SGD puro, permite paralelismo en GPU y generaliza mejor que batch completo (el ruido actúa como regularizador). El batch efectivo es B × gradient_accumulation_steps."}
  ]},
  {"domain":"machine_learning","messages":[
    {"role":"user","content":"¿Cuándo usar Random Forest vs Gradient Boosting?"},
    {"role":"assistant","content":
     "Random Forest: paralelo, robusto a outliers, fácil de tunear, bueno como baseline. Gradient Boosting (XGBoost, LightGBM): típicamente más preciso en tabular data, pero más sensible a hiperparámetros y sobreajuste. Regla práctica: si el tiempo es limitado y el dataset es ruidoso, RF; si se quiere maximizar AUC/RMSE en competencias con datos limpios, GBM con early stopping."}
  ]},
  # ── Python ────────────────────────────────────────────────────────────
  {"domain":"python","messages":[
    {"role":"user","content":"¿Qué es un generador en Python y para qué sirve?"},
    {"role":"assistant","content":
     "Un generador es una función con `yield` que produce valores uno a uno, sin cargar todo en memoria. Útil para datasets grandes, pipelines de procesamiento y secuencias infinitas. Ejemplo: `def naturales(): n=0; while True: yield n; n+=1`. Se itera con `next()` o en un `for`. Los generadores son objetos perezosos (lazy): el cuerpo solo se ejecuta hasta el próximo `yield`."}
  ]},
  {"domain":"python","messages":[
    {"role":"user","content":"Escribe una función Python para calcular la potencia aparente compleja."},
    {"role":"assistant","content":
     "```python\nimport cmath\n\ndef potencia_aparente(v_fasorial: complex, i_fasorial: complex) -> complex:\n    \"\"\"S = V · I* [VA]\"\"\"\n    S = v_fasorial * i_fasorial.conjugate()\n    return S\n\n# Ejemplo: V=220∠0°, I=10∠-30° (carga inductiva)\nV = 220 + 0j\nI = 10 * cmath.exp(-1j * cmath.pi/6)\nS = potencia_aparente(V, I)\nprint(f'P={S.real:.1f} W, Q={S.imag:.1f} VAR, |S|={abs(S):.1f} VA')\n```"}
  ]},
  {"domain":"python","messages":[
    {"role":"user","content":"¿Cómo funciona el decorador @property en Python?"},
    {"role":"assistant","content":
     "@property convierte un método en un atributo de solo lectura (o lectura-escritura con @setter). Permite encapsular lógica de validación sin cambiar la API. Ejemplo:\n```python\nclass Circulo:\n    def __init__(self, r): self._r = r\n    @property\n    def area(self): return 3.14159 * self._r**2\n    @property\n    def radio(self): return self._r\n    @radio.setter\n    def radio(self, v):\n        if v < 0: raise ValueError('radio negativo')\n        self._r = v\n```"}
  ]},
  # ── SQL / PostgreSQL ──────────────────────────────────────────────────
  {"domain":"sql","messages":[
    {"role":"user","content":"¿Cómo optimizar una consulta SQL lenta en PostgreSQL?"},
    {"role":"assistant","content":
     "1) `EXPLAIN ANALYZE` para ver el plan de ejecución y cuellos de botella (Seq Scan costoso). 2) Agregar índice en columnas de filtros/JOINs frecuentes: `CREATE INDEX idx ON tabla(col)`. 3) Evitar `SELECT *`; pedir solo columnas necesarias. 4) Usar CTEs materializadas con cuidado (en PG 12+ son transparentes por defecto). 5) Actualizar estadísticas: `ANALYZE tabla`. 6) Considerar índices parciales, índices de cobertura (INCLUDE) o particionamiento para tablas muy grandes."}
  ]},
  {"domain":"sql","messages":[
    {"role":"user","content":"¿Qué diferencia hay entre INNER JOIN, LEFT JOIN y FULL OUTER JOIN?"},
    {"role":"assistant","content":
     "INNER JOIN: solo filas con coincidencia en AMBAS tablas. LEFT JOIN: todas las filas de la tabla izquierda; NULLs donde no hay coincidencia en la derecha. RIGHT JOIN: lo inverso. FULL OUTER JOIN: todas las filas de ambas tablas; NULLs donde no hay coincidencia en ninguno de los lados. Regla: usa LEFT JOIN cuando la tabla izquierda es la 'maestra' y quieres conservar todos sus registros."}
  ]},
  {"domain":"sql","messages":[
    {"role":"user","content":"¿Cuándo usar índice B-tree vs GIN vs GiST en PostgreSQL?"},
    {"role":"assistant","content":
     "B-tree (defecto): búsquedas de igualdad y rango en tipos comparables (enteros, texto, fechas). GIN: búsqueda en arrays, JSONB, o full-text search (tsvector); ideal para `@>` y `@@`. GiST: datos geométricos/geoespaciales (PostGIS), rangos superpuestos, búsqueda por similitud de texto (pg_trgm). BRIN: tablas muy grandes con correlación física (ej. series de tiempo insertadas en orden); muy pequeño pero solo para rangos."}
  ]},
  # ── AWS ───────────────────────────────────────────────────────────────
  {"domain":"aws","messages":[
    {"role":"user","content":"¿Cómo diseñarías una arquitectura serverless para procesar 1 millón de eventos/hora?"},
    {"role":"assistant","content":
     "API Gateway → Lambda (ingestión ligera) → SQS FIFO o Kinesis (buffer) → Lambda o Fargate (procesamiento) → DynamoDB o S3+Glue (almacenamiento). SQS desacopla productores y consumidores; Kinesis es mejor si necesitas orden y replay. Lambda escala automáticamente hasta 1 000 concurrencias por defecto (aumentable). Para >100 MB/s considera Kinesis + Flink en EMR. Monitorear con CloudWatch Metrics y alarmas en DLQ (Dead Letter Queue)."}
  ]},
  {"domain":"aws","messages":[
    {"role":"user","content":"¿Qué es AWS S3 Intelligent-Tiering y cuándo usarlo?"},
    {"role":"assistant","content":
     "S3 Intelligent-Tiering mueve objetos automáticamente entre capas de acceso (frecuente/infrecuente/archivo) según el patrón de acceso, sin overhead de recuperación. Útil cuando el patrón de acceso es impredecible: logs que se consultan mucho el primer mes y luego casi nunca. Costo: pequeña tarifa de monitoreo por objeto (~0.0025 USD/1000 obj/mes). No usar para objetos <128 KB (no se mueven) ni para accesos muy predecibles (mejor usar clases manuales)."}
  ]},
  # ── Matemáticas / Álgebra Lineal ──────────────────────────────────────
  {"domain":"matematicas","messages":[
    {"role":"user","content":"Explica la descomposición SVD y sus aplicaciones en ML."},
    {"role":"assistant","content":
     "SVD: A = U·Σ·Vᵀ, donde U (m×m) y V (n×n) son ortogonales y Σ (m×n) contiene valores singulares σ₁≥σ₂≥…≥0. Aplicaciones: compresión (truncar a rango k preserva máxima energía), PCA (SVD de la matriz centrada), sistemas de recomendación (matrix factorization), pseudoinversa A⁺=VΣ⁺Uᵀ, resolución de sistemas sobredeterminados por mínimos cuadrados."}
  ]},
  {"domain":"matematicas","messages":[
    {"role":"user","content":"¿Qué es el gradiente de una función escalar y cómo se calcula para f(x,y)=x²y+sin(y)?"},
    {"role":"assistant","content":
     "El gradiente ∇f es el vector de derivadas parciales: ∇f = (∂f/∂x, ∂f/∂y). Para f(x,y)=x²y+sin(y): ∂f/∂x = 2xy; ∂f/∂y = x² + cos(y). Por ejemplo en (1, π/2): ∇f = (2·1·π/2, 1+cos(π/2)) = (π, 1). El gradiente apunta en la dirección de máximo crecimiento de f; el descenso de gradiente va en la dirección opuesta."}
  ]},
  {"domain":"matematicas","messages":[
    {"role":"user","content":"¿Qué son los autovalores y autovectores de una matriz?"},
    {"role":"assistant","content":
     "Para una matriz A, el autovector v y autovalor λ satisfacen Av = λv: la transformación solo escala v, sin cambiar su dirección. Se calculan resolviendo det(A - λI)=0 (polinomio característico). Aplicaciones: PCA (autovectores de la covarianza = componentes principales), análisis de estabilidad en control (autovalores de la matriz de sistema), PageRank (autovector dominante de la matriz de adyacencia)."}
  ]},
  # ── Probabilidad / Estadística ────────────────────────────────────────
  {"domain":"probabilidad","messages":[
    {"role":"user","content":"Explica el Teorema de Bayes con un ejemplo concreto."},
    {"role":"assistant","content":
     "P(A|B) = P(B|A)·P(A) / P(B). Ejemplo: test médico con sensibilidad 95% y especificidad 90% para enfermedad con prevalencia 1%. Si el test es positivo, P(enfermo|+) = (0.95·0.01) / (0.95·0.01 + 0.10·0.99) ≈ 8.7%. Aunque el test es bueno, la baja prevalencia hace que la mayoría de positivos sean falsos positivos. Esto ilustra por qué el prior (prevalencia) importa tanto como la verosimilitud."}
  ]},
  {"domain":"probabilidad","messages":[
    {"role":"user","content":"¿Qué es la divergencia KL y para qué se usa en ML?"},
    {"role":"assistant","content":
     "KL(P||Q) = Σ P(x) log(P(x)/Q(x)) mide cuánta información extra se necesita para codificar muestras de P usando el código óptimo de Q. No es simétrica: KL(P||Q)≠KL(Q||P). En ML: función de pérdida en VAEs, regularización en RLHF (penalizar alejarse de la política SFT), entrenamiento de modelos de lenguaje (minimizar KL entre distribución del modelo y los datos). KL=0 sii P=Q."}
  ]},
  # ── Redes neuronales / Deep Learning ─────────────────────────────────
  {"domain":"deep_learning","messages":[
    {"role":"user","content":"¿Qué es el problema del gradiente que desaparece y cómo se soluciona?"},
    {"role":"assistant","content":
     "En redes profundas con sigmoide/tanh, los gradientes se multiplican por derivadas <1 en cada capa → tienden a 0 en capas tempranas → no aprenden. Soluciones: (1) ReLU (derivada=1 para x>0); (2) batch normalization (estabiliza distribuciones intermedias); (3) skip connections (ResNet suma el input al output, gradiente fluye directamente); (4) inicialización cuidadosa (He, Xavier); (5) gradient clipping para evitar explosión."}
  ]},
  {"domain":"deep_learning","messages":[
    {"role":"user","content":"¿Cómo funciona la normalización por lotes (Batch Normalization)?"},
    {"role":"assistant","content":
     "BN normaliza las activaciones de una capa dentro de cada mini-batch: x̂ = (x-μ_B)/√(σ²_B+ε), luego escala y desplaza: y = γ·x̂ + β (γ, β aprendibles). Beneficios: permite tasas de aprendizaje más altas, reduce sensibilidad a inicialización, actúa como regularizador suave. En inferencia usa media/varianza acumuladas (running stats). Alternativa cuando el batch es pequeño: Layer Norm (sobre las features de cada ejemplo, estándar en transformers)."}
  ]},
  # ── Transformers / NLP ────────────────────────────────────────────────
  {"domain":"transformers","messages":[
    {"role":"user","content":"Explica el mecanismo de atención multi-cabeza (multi-head attention)."},
    {"role":"assistant","content":
     "MHA proyecta Q, K, V a h sub-espacios paralelos (cabezas), calcula atención escalada en cada uno: Attn(Q,K,V) = softmax(QKᵀ/√d_k)·V, y concatena las salidas. Cada cabeza puede atender a diferentes patrones (sintaxis, semántica, co-referencia). La proyección final W_O mezcla las cabezas. Costo: O(T²·d_model) en memoria y tiempo por la matriz QKᵀ — cuello de botella para secuencias largas, motivando Flash Attention y atención sparse."}
  ]},
  {"domain":"transformers","messages":[
    {"role":"user","content":"¿Qué son los embeddings posicionales RoPE y por qué son preferidos hoy?"},
    {"role":"assistant","content":
     "RoPE (Rotary Position Embedding) codifica la posición rotando el vector de query/key en un ángulo proporcional a su posición. A diferencia de los embeddings absolutos aprendidos, el producto interno Q·Kᵀ solo depende de la posición relativa (j-i), lo que facilita la generalización a longitudes de contexto mayores que las vistas en entrenamiento. Usado en LLaMA, Mistral, Qwen, SmolLM2. Con 'rope scaling' (NTK, YaRN) se puede extender el contexto 4-16× post-entrenamiento."}
  ]},
  # ── Series de tiempo ──────────────────────────────────────────────────
  {"domain":"series_tiempo","messages":[
    {"role":"user","content":"¿Qué es la estacionalidad en series de tiempo y cómo detectarla?"},
    {"role":"assistant","content":
     "Estacionalidad: patrón que se repite con período fijo (diario, semanal, anual). Detección: (1) gráfico de la serie y descomposición STL (Seasonal-Trend usando Loess); (2) autocorrelación ACF: picos en múltiplos del período; (3) periodograma (FFT): picos en frecuencias estacionales. En demanda eléctrica hay estacionalidad diaria (curva de carga) y semanal (hábitos laborales/fin de semana)."}
  ]},
  {"domain":"series_tiempo","messages":[
    {"role":"user","content":"¿Cuándo usar ARIMA vs modelos de ML para pronóstico?"},
    {"role":"assistant","content":
     "ARIMA: bueno con series estacionarias, cortos horizontes, pocos datos, cuando se necesita interpretabilidad y bandas de confianza analíticas. ML (XGBoost, LSTM, Transformer): captura no-linealidades y múltiples variables exógenas; requiere más datos y validación temporal cuidadosa. Regla práctica: comenzar con ARIMA/SARIMA/ETS como baseline; si el error es alto o hay muchas covariables, escalar a ML. M4/M5 Competition: métodos híbridos (ARIMA+ML) frecuentemente ganan."}
  ]},
  # ── MLOps / Docker ────────────────────────────────────────────────────
  {"domain":"mlops","messages":[
    {"role":"user","content":"¿Qué es el data drift y cómo monitorearlo en producción?"},
    {"role":"assistant","content":
     "Data drift: cambio en la distribución de los datos de entrada (P(X) cambia) que degrada el modelo aunque la relación P(Y|X) siga igual. Detección: test KS (Kolmogorov-Smirnov) o PSI (Population Stability Index) entre la distribución de referencia y la ventana reciente. Herramientas: Evidently, WhyLogs, Grafana+Prometheus. Alerta si PSI > 0.2 (cambio mayor) o KS p-value < 0.05. Reacción: reentrenar, recalibrar, o alertar al equipo."}
  ]},
  {"domain":"mlops","messages":[
    {"role":"user","content":"¿Cuál es la diferencia entre Docker image y Docker container?"},
    {"role":"assistant","content":
     "Image: plantilla inmutable con el sistema de archivos (capas de solo lectura). Container: instancia en ejecución de una imagen, con una capa de escritura efímera. Una imagen puede generar múltiples containers. Analogía: la imagen es la clase, el container es el objeto. Los containers se crean con `docker run`, se detienen con `docker stop`, y su estado efímero se pierde al eliminarlos (a menos que uses volúmenes montados)."}
  ]},
  # ── Optimización ──────────────────────────────────────────────────────
  {"domain":"optimizacion","messages":[
    {"role":"user","content":"Compara Adam, AdamW y SGD con momentum para entrenar LLMs."},
    {"role":"assistant","content":
     "SGD+momentum: rápido por paso, requiere LR muy ajustado, generaliza bien en algunas arquitecturas. Adam: adapta la LR por parámetro (media/varianza del gradiente), converge rápido, pero con weight decay regular contamina la adaptación → AdamW. AdamW (Loshchilov & Hutter): desacopla weight decay de la actualización adaptativa, estándar para LLMs. Ajuste típico: lr=3e-4, β1=0.9, β2=0.95 (no 0.999 como Adam clásico), weight_decay=0.1, grad_clip=1.0."}
  ]},
  {"domain":"optimizacion","messages":[
    {"role":"user","content":"¿Qué es el learning rate scheduling y qué scheduler usar?"},
    {"role":"assistant","content":
     "El scheduler varía la LR durante el entrenamiento. Cosine annealing: LR decrece suavemente de lr_max a lr_min siguiendo un coseno; estándar en LLMs (GPT, LLaMA). Warmup: primeras N iteraciones LR sube linealmente desde 0 hasta lr_max; evita gradientes explosivos al inicio. Cyclical LR: oscila para escapar mínimos locales. OneCycleLR de PyTorch: warmup + cosine decay en un solo ciclo. Regla: warmup_steps ≈ 1–5% del total."}
  ]},
  # ── Redes y Control ───────────────────────────────────────────────────
  {"domain":"control","messages":[
    {"role":"user","content":"¿Qué es un controlador PID y cómo se sintoniza?"},
    {"role":"assistant","content":
     "PID: u(t) = Kp·e(t) + Ki·∫e dt + Kd·de/dt. Kp reduce el error estacionario, Ki lo elimina, Kd amortigua oscilaciones. Sintonía Ziegler-Nichols: encontrar Ku (ganancia última que causa oscilación sostenida) y Tu (período); luego Kp=0.6Ku, Ki=2Kp/Tu, Kd=KpTu/8. En la práctica se usa autotuning (relay feedback) o ajuste manual iterativo. Para sistemas digitales, la frecuencia de muestreo debe ser ≥10× la frecuencia de corte del lazo."}
  ]},
  {"domain":"electronica_potencia","messages":[
    {"role":"user","content":"¿Cómo funciona un convertidor DC-DC buck (reductor)?"},
    {"role":"assistant","content":
     "El buck regula V_out < V_in mediante un interruptor (MOSFET) con ciclo de trabajo D=V_out/V_in. Cuando el switch está ON, la energía fluye a través del inductor hacia la carga; cuando está OFF, el inductor mantiene la corriente a través del diodo (o switch síncrono). El condensador filtra el rizado de voltaje. En modo discontinuo (DCM) el inductor se vacía completamente; en CCM, la corriente nunca llega a cero. Eficiencias típicas: 85–98%."}
  ]},
  # ── Git ───────────────────────────────────────────────────────────────
  {"domain":"git","messages":[
    {"role":"user","content":"¿Cuál es la diferencia entre git rebase y git merge?"},
    {"role":"assistant","content":
     "Merge: crea un commit de fusión que une dos historias, preservando el historial completo (no-lineal). Rebase: mueve o 'replanta' commits sobre otra base, reescribiendo sus hashes → historial lineal y limpio. Regla: no hacer rebase de ramas públicas compartidas (reescribe historia). Flujo habitual: `git pull --rebase` para mantener historial limpio en ramas personales; `merge` para integrar feature branches a main con registro de cuándo ocurrió la fusión."}
  ]},
]

print(f"Ejemplos únicos: {len(SFT_RAW)} en {len(set(e['domain'] for e in SFT_RAW))} dominios")

In [ ]:
# ── Estadísticas del dataset ──────────────────────────────────────────────

# Longitudes de respuestas
resp_lens = [len(e["messages"][-1]["content"]) for e in SFT_RAW]
tokens_approx = [l // 4 for l in resp_lens]  # estimación naive (4 chars/token)

print(f"Respuestas — longitud en chars:  min={min(resp_lens):3d}  med={int(np.median(resp_lens)):3d}  max={max(resp_lens):3d}")
print(f"Tokens aprox (÷4):               min={min(tokens_approx):3d}  med={int(np.median(tokens_approx)):3d}  max={max(tokens_approx):3d}")

# Distribución por dominio
domain_count = collections.Counter(e["domain"] for e in SFT_RAW)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico de barras por dominio
ax = axes[0]
domains = list(domain_count.keys())
counts  = list(domain_count.values())
colors  = plt.cm.tab20(np.linspace(0, 1, len(domains)))
ax.barh(domains, counts, color=colors)
ax.set_xlabel("Ejemplos"); ax.set_title("Distribución por dominio")
ax.invert_yaxis()

# Histograma de longitudes
ax = axes[1]
ax.hist(resp_lens, bins=15, color="#5b8dd9", edgecolor="white")
ax.axvline(np.median(resp_lens), color="red", ls="--", label=f"mediana={int(np.median(resp_lens))} chars")
ax.set_xlabel("Longitud de respuesta (chars)"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de longitudes de respuesta"); ax.legend()

plt.tight_layout(); plt.show()

## 4. Chat templates

El modelo solo ve **tokens**: los roles son una convención materializada con tokens
especiales. El formato **ChatML** es el más extendido hoy:

```
<|im_start|>system
Eres un asistente técnico.<|im_end|>
<|im_start|>user
¿Qué es LoRA?<|im_end|>
<|im_start|>assistant
LoRA es...<|im_end|>
```

> ⚠️ **Template mismatch**: si en inferencia usas un formato distinto al visto en SFT,
> el rendimiento cae drásticamente aunque los pesos sean los mismos.

In [ ]:
IM_START, IM_END = "<|im_start|>", "<|im_end|>"

def format_chat(messages, add_generation_prompt=False):
    """Serializa una lista de messages al formato ChatML."""
    out = ""
    for m in messages:
        out += f"{IM_START}{m['role']}\n{m['content']}{IM_END}\n"
    if add_generation_prompt:
        out += f"{IM_START}assistant\n"
    return out

# Mostrar un ejemplo
ex_msgs = SFT_RAW[0]["messages"]
print("=== Texto completo (para entrenamiento) ===")
print(format_chat(ex_msgs))
print("=== Con prompt de generación (para inferencia) ===")
print(format_chat(ex_msgs[:1], add_generation_prompt=True))

In [ ]:
# Comparar templates: ChatML vs Alpaca vs Llama-3
def format_alpaca(messages):
    user = next(m["content"] for m in messages if m["role"] == "user")
    asst = next((m["content"] for m in messages if m["role"] == "assistant"), "")
    return f"### Instruction:\n{user}\n\n### Response:\n{asst}"

def format_llama3(messages):
    out = "<|begin_of_text|>"
    for m in messages:
        role_tag = f"<|start_header_id|>{m['role']}<|end_header_id|>"
        out += f"{role_tag}\n\n{m['content']}<|eot_id|>"
    out += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return out

msg = SFT_RAW[1]["messages"]
for name, fn in [("ChatML", format_chat), ("Alpaca", format_alpaca), ("Llama-3", format_llama3)]:
    print(f"── {name} ──")
    print(fn(msg) if name != "ChatML" else fn(msg, add_generation_prompt=False))
    print()

## 5. SFT con masking del prompt

Solo queremos que el modelo aprenda a **generar la respuesta**, no a regenerar la
instrucción. Los tokens del prompt se enmascaran con `-100` (el `ignore_index` de
`F.cross_entropy`): su contribución al gradiente es cero.

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|R|}\sum_{t \in \text{respuesta}} \log P_\theta(x_t \mid x_{<t})$$

donde $|R|$ es el número de tokens de respuesta y los tokens del prompt no
aparecen en la suma.

In [ ]:
# ── Tokenizador de caracteres para ilustrar masking de forma ejecutable ──

def build_char_vocab(texts):
    chars = sorted(set("".join(texts)))
    return {c: i for i, c in enumerate(chars)}, chars

# Construir vocabulario con todos los ejemplos
all_texts = [format_chat(e["messages"]) for e in SFT_RAW]
stoi, itos = build_char_vocab(all_texts)
encode = lambda t: [stoi[c] for c in t if c in stoi]
decode = lambda ids: "".join(itos[i] for i in ids)

def build_sft_example(messages, encode_fn):
    """Devuelve (input_ids, labels) con -100 en posiciones del prompt."""
    prompt = format_chat(messages[:-1], add_generation_prompt=True)
    full   = format_chat(messages, add_generation_prompt=False)
    p_ids  = encode_fn(prompt)
    f_ids  = encode_fn(full)
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return f_ids, labels

ex = SFT_RAW[0]["messages"]
ids, labels = build_sft_example(ex, encode)
mask = np.array([0 if l == -100 else 1 for l in labels])

print(f"Total tokens  : {len(ids)}")
print(f"Prompt (mask) : {(mask == 0).sum()}  tokens ignorados en la loss")
print(f"Respuesta     : {(mask == 1).sum()}  tokens que contribuyen a la loss")

plt.figure(figsize=(13, 1.8))
plt.imshow(mask[None, :], aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
plt.yticks([]); plt.xlabel("Posición del token")
plt.title("Máscara de loss  — rojo=prompt (ignorado)  verde=respuesta (entrena)")
plt.colorbar(shrink=0.5, ticks=[0,1], label="0=ignorado  1=entrena")
plt.tight_layout(); plt.show()

In [ ]:
# ── Implementación de la loss enmascarada ──────────────────────────────

def masked_cross_entropy(logits, labels, ignore_index=-100):
    """
    logits: (T, V) — scores del modelo en cada posición
    labels: (T,)  — índices objetivo, -100 donde no se entrena
    """
    return F.cross_entropy(
        logits,
        torch.tensor(labels, dtype=torch.long),
        ignore_index=ignore_index
    )

# Verificar con logits aleatorios
V = len(stoi)
T = len(ids)
torch.manual_seed(SEED)
fake_logits = torch.randn(T, V)

loss_masked   = masked_cross_entropy(fake_logits, labels)
labels_full   = list(ids[1:]) + [0]        # siguiente token en cada posición
loss_unmasked = masked_cross_entropy(fake_logits, labels_full)

print(f"Loss enmascarada (solo respuesta) : {loss_masked.item():.4f}")
print(f"Loss sin máscara (prompt+respuesta): {loss_unmasked.item():.4f}")
print(f"Ratio: {loss_unmasked.item()/loss_masked.item():.2f}x")
print("\nEl ratio > 1 es normal: la loss enmascarada solo promedia sobre tokens de respuesta.")

## 6. LoRA desde cero

### 6.1 La hipótesis del rango bajo

**Hu et al. (2021)** conjeturan que la actualización $\Delta W$ del fine-tuning tiene
**rango intrínseco bajo** (los gradientes de la tarea "viven" en un subespacio
pequeño).  En vez de actualizar $W \in \mathbb{R}^{d \times k}$ directamente:

$$W' = W + \Delta W = W + \underbrace{\frac{\alpha}{r} B A}_{\text{rango } r},
\quad A \in \mathbb{R}^{r \times k},\ B \in \mathbb{R}^{d \times r},\quad r \ll \min(d,k)$$

- $W$ se **congela**: sus gradientes no se calculan.
- $B$ se inicializa en **cero** → al inicio $\Delta W = 0$ (no perturba el modelo).
- Solo $A$ y $B$ se entrenan: $r(d+k)$ params vs $dk$ en FT completo.
- $\alpha/r$ es un factor de escala; convención habitual: $\alpha = 2r$.

### 6.2 Reducción de parámetros

$$\text{ratio} = \frac{r(d+k)}{dk} = r\left(\frac{1}{k}+\frac{1}{d}\right)$$

Para $d=k=4096$ y $r=8$: ratio $\approx 0.4\%$ de los parámetros de esa capa.

In [ ]:
# ── LoRALinear: implementación mínima pero completa ──────────────────────

class LoRALinear(nn.Module):
    """
    Envuelve una nn.Linear con adaptadores LoRA de rango r.
    El peso base se congela; solo A y B son entrenables.
    """
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base    = base_linear
        self.r       = r
        self.scaling = alpha / r

        # Congelar pesos base
        for p in self.base.parameters():
            p.requires_grad = False

        d_out = base_linear.out_features
        d_in  = base_linear.in_features

        # A: inicialización kaiming (activación varianza 1)
        # B: inicialización cero → ΔW=0 al inicio
        self.A = nn.Parameter(torch.empty(r, d_in))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., d_in)
        base_out = self.base(x)                      # (..., d_out)
        lora_out = (x @ self.A.T) @ self.B.T         # (..., d_out)
        return base_out + self.scaling * lora_out

    @property
    def delta_W(self) -> torch.Tensor:
        """ΔW = (α/r) · B · A   shape: (d_out, d_in)"""
        return self.scaling * (self.B @ self.A)

    def merge(self) -> nn.Linear:
        """Fusiona LoRA en el peso base → nn.Linear sin overhead."""
        merged = nn.Linear(self.base.in_features, self.base.out_features,
                           bias=self.base.bias is not None)
        merged.weight = nn.Parameter(self.base.weight.data + self.delta_W.detach())
        if self.base.bias is not None:
            merged.bias = nn.Parameter(self.base.bias.data.clone())
        return merged

# Test rápido
torch.manual_seed(SEED)
lin  = nn.Linear(64, 32, bias=False)
lora = LoRALinear(lin, r=4, alpha=8.0)
x    = torch.randn(8, 64)

# Al inicio ΔW=0 → salidas idénticas
with torch.no_grad():
    y_base = lin(x); y_lora = lora(x)
assert torch.allclose(y_base, y_lora), "ΔW inicial debe ser cero"
print("OK: al inicio LoRA no perturba el modelo (B=0, ΔW=0)")

# Verificar parámetros entrenables
total     = sum(p.numel() for p in lora.parameters())
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"d_out={lin.out_features}  d_in={lin.in_features}  r=4")
print(f"Params: total={total}  entrenables={trainable}  ratio={100*trainable/total:.1f}%")

### 6.3 MiniGPT toy — decodificador de lenguaje

Definimos un decodificador compacto para inyectar LoRA y **entrenar de verdad**
sobre una tarea de lenguaje, mostrando la curva de loss.

In [ ]:
# ── MiniGPT: decoder-only transformer toy (~100 líneas) ─────────────────

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_len: int = 128):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.qkv      = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj     = nn.Linear(d_model, d_model, bias=False)
        # Máscara causal
        mask = torch.tril(torch.ones(max_len, max_len)).unsqueeze(0).unsqueeze(0)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        # Reshape a (B, n_heads, T, head_dim)
        def reshape(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k, v = reshape(q), reshape(k), reshape(v)
        scale  = self.head_dim ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale          # (B, H, T, T)
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        attn   = torch.softmax(scores, dim=-1)
        out    = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class MiniGPTBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, max_len: int = 128):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff1  = nn.Linear(d_model, d_ff, bias=False)
        self.ff2  = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff2(torch.relu(self.ff1(self.ln2(x))))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size=64, d_model=64, n_layers=2, n_heads=4, d_ff=128, max_len=128):
        super().__init__()
        self.tok_emb  = nn.Embedding(vocab_size, d_model)
        self.pos_emb  = nn.Embedding(max_len, d_model)
        self.blocks   = nn.ModuleList([
            MiniGPTBlock(d_model, n_heads, d_ff, max_len) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):                          # idx: (B, T)
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        x    = self.tok_emb(idx) + self.pos_emb(pos)  # (B, T, d_model)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_final(x))          # (B, T, vocab_size)

    def generate(self, idx, max_new=20, temperature=1.0):
        for _ in range(max_new):
            logits = self(idx[:, -self.pos_emb.num_embeddings:])[:, -1, :] / temperature
            next_tok = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx

# Instanciar modelo toy
torch.manual_seed(SEED)
mini_gpt = MiniGPT(vocab_size=len(stoi), d_model=64, n_layers=2, n_heads=4, d_ff=128)
total_p  = sum(p.numel() for p in mini_gpt.parameters())
print(f"MiniGPT: {total_p:,} parámetros  vocab={len(stoi)}")

In [ ]:
# ── Inyectar LoRA en MiniGPT ─────────────────────────────────────────────

def inject_lora(model: nn.Module, target_modules=("qkv", "proj"), r=4, alpha=8.0):
    """Reemplaza nn.Linear indicadas con LoRALinear in-place."""
    for name, module in list(model.named_modules()):
        for attr_name in dir(module):
            if not any(attr_name == t or attr_name.endswith("_" + t) for t in target_modules):
                continue
            child = getattr(module, attr_name, None)
            if isinstance(child, nn.Linear):
                setattr(module, attr_name, LoRALinear(child, r=r, alpha=alpha))
    return model

torch.manual_seed(SEED)
mini_gpt = MiniGPT(vocab_size=len(stoi), d_model=64, n_layers=2, n_heads=4, d_ff=128)
mini_gpt = inject_lora(mini_gpt, target_modules=("qkv", "proj", "ff1", "ff2"), r=4, alpha=8.0)

total_p     = sum(p.numel() for p in mini_gpt.parameters())
trainable_p = sum(p.numel() for p in mini_gpt.parameters() if p.requires_grad)
print(f"Después de LoRA:")
print(f"  Total       : {total_p:,}")
print(f"  Entrenables : {trainable_p:,}  ({100*trainable_p/total_p:.2f}%)")

In [ ]:
# ── Preparar corpus de entrenamiento (SFT toy) ───────────────────────────

# Serializar todos los ejemplos SFT a texto
corpus_text = "".join(format_chat(e["messages"]) for e in SFT_RAW)
corpus_ids  = encode(corpus_text)
print(f"Corpus: {len(corpus_text):,} chars  →  {len(corpus_ids):,} tokens (char-level)")

# Dataset: ventanas de largo T con máscara de prompt (simplificada a labels == input shift)
T_CTX = 64   # contexto para el modelo toy

def make_batches(ids, ctx, batch_size, n_batches, seed=42):
    rng = np.random.default_rng(seed)
    max_start = len(ids) - ctx - 1
    for _ in range(n_batches):
        starts = rng.integers(0, max_start, size=batch_size)
        x_batch = torch.tensor([ids[s:s+ctx] for s in starts], dtype=torch.long)
        y_batch = torch.tensor([ids[s+1:s+ctx+1] for s in starts], dtype=torch.long)
        yield x_batch, y_batch

# Verificar que el modelo forward funciona
x_test, y_test = next(make_batches(corpus_ids, T_CTX, 4, 1))
with torch.no_grad():
    logits = mini_gpt(x_test)  # (4, T_CTX, vocab_size)
print(f"Forward OK: logits shape = {tuple(logits.shape)}")

In [ ]:
# ── Entrenamiento LoRA sobre el corpus SFT ───────────────────────────────

TRAIN_STEPS = 200
BATCH_SIZE  = 16
LR_LORA     = 5e-3

optimizer = torch.optim.AdamW(
    [p for p in mini_gpt.parameters() if p.requires_grad],
    lr=LR_LORA, weight_decay=0.01
)

lora_losses = []
mini_gpt.train()
for step, (xb, yb) in enumerate(make_batches(corpus_ids, T_CTX, BATCH_SIZE, TRAIN_STEPS, seed=SEED)):
    logits = mini_gpt(xb)                           # (B, T, V)
    loss   = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        yb.view(-1)
    )
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    lora_losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step+1:3d}/{TRAIN_STEPS}  loss={loss.item():.4f}")

print(f"\nFinal: {lora_losses[0]:.4f} → {lora_losses[-1]:.4f}  "
      f"({100*(lora_losses[0]-lora_losses[-1])/lora_losses[0]:.1f}% mejora)")

In [ ]:
# ── Curva de entrenamiento LoRA ───────────────────────────────────────────

smooth = np.convolve(lora_losses, np.ones(10)/10, mode="valid")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lora_losses, alpha=0.3, color="#5b8dd9", label="loss por step")
ax.plot(np.arange(9, len(lora_losses)), smooth, color="#1a4a8a", lw=2, label="media móvil 10")
ax.set_xlabel("Step"); ax.set_ylabel("Cross-entropy loss")
ax.set_title("Entrenamiento LoRA sobre MiniGPT (corpus SFT toy)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── % parámetros entrenables vs rango r ─────────────────────────────────

rs   = [1, 2, 4, 8, 16, 32, 64]
pcts = []
for r_val in rs:
    m_tmp = MiniGPT(vocab_size=len(stoi), d_model=64, n_layers=2, n_heads=4, d_ff=128)
    m_tmp = inject_lora(m_tmp, target_modules=("qkv", "proj", "ff1", "ff2"), r=r_val, alpha=float(r_val*2))
    tr = sum(p.numel() for p in m_tmp.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m_tmp.parameters())
    pcts.append(100 * tr / tot)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rs, pcts, "o-", color="#e07070", lw=2)
for r_val, p in zip(rs, pcts):
    ax.annotate(f"{p:.1f}%", (r_val, p), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xlabel("Rango r"); ax.set_ylabel("% parámetros entrenables")
ax.set_title("LoRA: parámetros entrenables vs rango  (MiniGPT 64d, 2 capas)")
ax.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

print("\nTable: r → % entrenables")
for r_val, p in zip(rs, pcts):
    print(f"  r={r_val:3d}  {p:.2f}%")

## 7. LoRA con la librería PEFT

En producción se usa `peft` de HuggingFace. `LoraConfig` especifica:
- `r`, `lora_alpha`, `lora_dropout`
- `target_modules`: por nombre o lista (ej. `["q_proj", "v_proj"]`)
- `task_type`: `CAUSAL_LM` para LLMs

Después del entrenamiento, `merge_and_unload()` fusiona $\Delta W$ en $W$ para
inferencia sin overhead de LoRA.

In [ ]:
# ── PEFT con modelo HF tiny (try/except con fallback) ────────────────────

try:
    pipq("transformers", "peft", "accelerate")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model, TaskType

    # Intentar modelo del perfil, caer a tiny-gpt2
    model_name = C["fallback_model"]
    print(f"Cargando {model_name} ...")
    hf_tok   = AutoTokenizer.from_pretrained(model_name)
    hf_model = AutoModelForCausalLM.from_pretrained(model_name)

    lora_cfg = LoraConfig(
        r             = C["lora_r"],
        lora_alpha    = C["lora_alpha"],
        lora_dropout  = C["lora_dropout"],
        target_modules= ["c_attn"],   # capas de atención en GPT-2
        task_type     = TaskType.CAUSAL_LM,
    )
    peft_model = get_peft_model(hf_model, lora_cfg)
    peft_model.print_trainable_parameters()

    # Merge and unload
    merged = peft_model.merge_and_unload()
    merged_p = sum(p.numel() for p in merged.parameters())
    print(f"\nDespués de merge_and_unload: {merged_p:,} params (sin overhead LoRA)")

except Exception as exc:
    print(f"[INFO] PEFT/HF no disponible: {type(exc).__name__}: {exc}")
    print("Usa la LoRA desde cero de la sección 6 (completamente funcional).")

## 8. QLoRA

**QLoRA** (Dettmers et al. 2023) combina tres técnicas:

1. **NF4** (NormalFloat 4-bit): los 16 niveles de cuantización se ubican en los
   cuantiles de una $\mathcal{N}(0,1)$, minimizando el error para pesos con
   distribución normal.
2. **Double quantization**: cuantiza también las constantes de cuantización
   (~0.5 bits/param adicionales), ahorrando ~0.37 GB para un modelo de 65B.
3. **Paged optimizers**: usa la memoria unificada de NVIDIA para hacer *paging*
   de los estados del optimizador a CPU durante picos de memoria.

$$\text{VRAM QLoRA} \approx \underbrace{N \cdot 0.5 \text{ B}}_{\text{pesos NF4}} + \underbrace{2 \cdot r(d+k) \cdot L \cdot 2 \text{ B}}_{\text{adaptadores fp16}}$$

Permite fine-tunear modelos de **7B en una GPU de 16 GB** y **65B en 48 GB**.

In [ ]:
# ── Comparativa de memoria: FT completo / LoRA fp16 / QLoRA NF4 ──────────

configs_mem = [
    ("GPT-2 (117M)",  117e6),
    ("SmolLM2-135M",  135e6),
    ("Qwen2.5-1.5B",  1.5e9),
    ("LLaMA-3  7B",   7e9),
    ("LLaMA-3 13B",   13e9),
    ("LLaMA-3 70B",   70e9),
]

def ft_full(p):    return p * 16 / 1e9
def lora_fp16(p):  return p * 2  / 1e9   # base fp16 + adaptadores negligibles
def qlora_nf4(p):  return p * 0.5 / 1e9  # base 4-bit + adaptadores

names_m  = [n for n, _ in configs_mem]
full_m   = [ft_full(p)    for _, p in configs_mem]
lora_m   = [lora_fp16(p)  for _, p in configs_mem]
qlora_m  = [qlora_nf4(p)  for _, p in configs_mem]

x = np.arange(len(names_m)); w = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w,   full_m,   w, label="FT completo", color="#e07070")
ax.bar(x,       lora_m,   w, label="LoRA (base fp16)", color="#70b0e0")
ax.bar(x + w,   qlora_m,  w, label="QLoRA (NF4)", color="#70d0a0")
ax.set_xticks(x); ax.set_xticklabels(names_m, rotation=25, ha="right")
ax.set_ylabel("VRAM estimada (GB)")
ax.set_title("Comparativa de memoria: FT completo vs LoRA vs QLoRA")
for thresh, color, label in [(16,"gray","16 GB T4"), (24,"#888","24 GB L4"), (40,"black","40 GB A100")]:
    ax.axhline(thresh, color=color, ls="--", lw=0.8, label=label)
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# Código QLoRA (comentado, solo se ejecuta con CUDA+bitsandbytes)
print("""
# Código para ejecutar con CUDA:
# from transformers import BitsAndBytesConfig
# bnb_cfg = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_cfg)
# peft_model = get_peft_model(model, lora_cfg)
""")

## 9. Comparación Before / After

Tras el entrenamiento, analizamos los adaptadores LoRA:
1. Distribuciones de $A$, $B$ y $\Delta W = \frac{\alpha}{r}BA$ por capa.
2. Norma de Frobenius $\|\Delta W\|_F$ por capa (indica qué capas cambiaron más).
3. Generaciones before/after (con el MiniGPT toy).

In [ ]:
# ── Recopilar adaptadores de todas las capas LoRA ─────────────────────────

lora_layers = {}
for name, module in mini_gpt.named_modules():
    if isinstance(module, LoRALinear):
        lora_layers[name] = module

print(f"Capas LoRA encontradas: {len(lora_layers)}")
for n in lora_layers:
    m = lora_layers[n]
    dW = m.delta_W.detach()
    print(f"  {n:40s}  shape=({m.base.out_features},{m.base.in_features})  "
          f"‖ΔW‖_F={dW.norm().item():.4f}")

In [ ]:
# ── Histogramas A, B, ΔW y normas ─────────────────────────────────────────

layer_names  = list(lora_layers.keys())[:4]  # tomar hasta 4 capas para visualizar
nW_frobenius = []
all_names    = list(lora_layers.keys())

fig, axes = plt.subplots(len(layer_names), 3, figsize=(14, 3 * len(layer_names)))
if len(layer_names) == 1:
    axes = axes[np.newaxis, :]

for row, name in enumerate(layer_names):
    lm  = lora_layers[name]
    A   = lm.A.detach().numpy().ravel()
    B   = lm.B.detach().numpy().ravel()
    dW  = lm.delta_W.detach().numpy().ravel()

    axes[row, 0].hist(A, bins=40, color="#5b8dd9", alpha=0.8)
    axes[row, 0].set_title(f"{name}\nA"); axes[row, 0].set_ylabel("Frecuencia")

    axes[row, 1].hist(B, bins=40, color="#e07070", alpha=0.8)
    axes[row, 1].set_title(f"B")

    axes[row, 2].hist(dW, bins=40, color="#70d0a0", alpha=0.8)
    axes[row, 2].set_title(f"ΔW = (α/r)·BA")

plt.suptitle("Distribución de adaptadores LoRA por capa", y=1.01)
plt.tight_layout(); plt.show()

# Norma de ΔW por capa (todas)
frobenius = []
for name in all_names:
    dW = lora_layers[name].delta_W.detach()
    frobenius.append(dW.norm().item())

fig, ax = plt.subplots(figsize=(12, 4))
x_idx = np.arange(len(all_names))
ax.bar(x_idx, frobenius, color="#5b8dd9")
ax.set_xticks(x_idx)
ax.set_xticklabels([n.split(".")[-1] + f"\n{n.split('.')[1]}" for n in all_names],
                    rotation=45, ha="right", fontsize=7)
ax.set_ylabel("‖ΔW‖_F"); ax.set_title("Norma de Frobenius de ΔW por capa LoRA")
plt.tight_layout(); plt.show()

In [ ]:
# ── Generaciones before/after ─────────────────────────────────────────────

# Modelo base (sin LoRA) para comparar
torch.manual_seed(SEED)
mini_base = MiniGPT(vocab_size=len(stoi), d_model=64, n_layers=2, n_heads=4, d_ff=128)

def generate_text(model, seed_text, max_new=60, temperature=0.8):
    model.eval()
    ids = encode(seed_text)
    if len(ids) == 0:
        ids = [0]
    x = torch.tensor([ids], dtype=torch.long)
    with torch.no_grad():
        out = model.generate(x, max_new=max_new, temperature=temperature)
    return decode(out[0].tolist())

prompts = [
    format_chat([{"role": "user", "content": "¿Qué es LoRA?"}],
                add_generation_prompt=True)[:60],
    "El flujo de carga calcula",
]

print("=" * 65)
for prompt in prompts:
    print(f"\nPrompt: {repr(prompt[:40])}...")
    gen_base = generate_text(mini_base, prompt)
    gen_lora = generate_text(mini_gpt,  prompt)
    print(f"  BASE : {repr(gen_base[len(prompt):len(prompt)+80])}")
    print(f"  LoRA : {repr(gen_lora[len(prompt):len(prompt)+80])}")
print("=" * 65)
print("(Generación char-level toy; la diferencia cualitativa se ve mejor con un modelo real)")

## 10. RLHF — Reinforcement Learning from Human Feedback

### Pipeline completo

```mermaid
flowchart LR
    SFT[modelo SFT] --> POL[política π_θ]
    POL --> GEN[genera y₁, y₂] --> HUM[anotadores humanos]
    HUM --> RM[reward model r_φ]
    RM --> PPO[PPO update]
    POL -.KL penalización.- REF[π_ref congelado]
    PPO --> POL
```

### Reward model (Bradley-Terry)

Dado un par (preferida $y_w$, rechazada $y_l$):

$$P(y_w \succ y_l \mid x) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big)$$

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}\big[\log \sigma(r_\phi(x,y_w) - r_\phi(x,y_l))\big]$$

### Objetivo RLHF

$$\max_\pi \mathbb{E}_{y \sim \pi}\big[r(x,y)\big] - \beta\,\mathrm{KL}\big(\pi(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big)$$

La penalización KL evita que la política se aleje demasiado del modelo SFT
(mantiene fluidez, evita reward hacking).

In [ ]:
# ── Loss del Reward Model (Bradley-Terry) ────────────────────────────────

def reward_loss(r_w: torch.Tensor, r_l: torch.Tensor) -> torch.Tensor:
    """- E[log σ(r_w - r_l)]"""
    return -F.logsigmoid(r_w - r_l).mean()

# Casos de prueba
scenarios = [
    ("Buen RM (preferida >> rechazada)", torch.tensor([2.0, 1.5, 3.0]), torch.tensor([0.5, 0.5, 1.0])),
    ("RM neutro (sin diferencia)",       torch.tensor([1.0, 1.0, 1.0]), torch.tensor([1.0, 1.0, 1.0])),
    ("RM malo (preferida < rechazada)",  torch.tensor([0.5, 0.5, 1.0]), torch.tensor([2.0, 1.5, 3.0])),
]
print(f"{'Escenario':<40}  Loss RM")
print("-" * 55)
for name, r_w, r_l in scenarios:
    print(f"{name:<40}  {reward_loss(r_w, r_l).item():.4f}")

# Curva: loss RM vs margen de reward
margins = torch.linspace(-4, 4, 100)
losses  = [-F.logsigmoid(m).item() for m in margins]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(margins.numpy(), losses, color="#5b8dd9", lw=2)
ax.axvline(0, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("Margen r_w − r_l"); ax.set_ylabel("Loss RM")
ax.set_title("Loss Bradley-Terry vs margen de reward\n(mínimo en margen → ∞)")
plt.tight_layout(); plt.show()

## 11. DPO — Direct Preference Optimization

### Derivación desde RLHF

El objetivo RLHF con restricción KL tiene solución en forma cerrada:

$$\pi^*(y|x) = \frac{1}{Z(x)}\,\pi_{\text{ref}}(y|x)\,\exp\!\left(\frac{r(x,y)}{\beta}\right)$$

Despejando $r$:

$$r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta\log Z(x)$$

Sustituyendo en la loss Bradley-Terry (el $\log Z(x)$ se cancela porque aparece
igual en $y_w$ y $y_l$):

$$\boxed{\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}\!\left[\log\sigma\!\left(
\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} -
\beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]}$$

**Ventajas vs PPO**: no necesita reward model separado, no requiere RL on-policy,
estable y mucho más barato (solo necesita la política y la referencia).

In [ ]:
# ── Implementación de la loss DPO ────────────────────────────────────────

def dpo_loss(
    log_pi_w:   torch.Tensor,   # log π_θ(y_w|x)
    log_pi_l:   torch.Tensor,   # log π_θ(y_l|x)
    log_ref_w:  torch.Tensor,   # log π_ref(y_w|x)
    log_ref_l:  torch.Tensor,   # log π_ref(y_l|x)
    beta: float = 0.1,
) -> torch.Tensor:
    """
    Loss DPO (Rafailov et al. 2023).
    Todos los tensores tienen shape (B,) — log-prob de la secuencia completa.
    """
    pi_logratios  = log_pi_w  - log_pi_l    # log(π_θ(y_w)/π_θ(y_l))
    ref_logratios = log_ref_w - log_ref_l   # log(π_ref(y_w)/π_ref(y_l))
    logits        = beta * (pi_logratios - ref_logratios)
    return -F.logsigmoid(logits).mean()

# ── Verificación numérica en 5 escenarios ──────────────────────────────────

print(f"{'Escenario':<45}  DPO loss")
print("-" * 60)
scenarios_dpo = [
    # (descripción, log_pi_w, log_pi_l, log_ref_w, log_ref_l)
    ("Política perfectamente alineada",
     torch.tensor([-1.0]), torch.tensor([-5.0]),
     torch.tensor([-3.0]), torch.tensor([-3.0])),
    ("Política igual a referencia (alineada)",
     torch.tensor([-2.5]), torch.tensor([-4.0]),
     torch.tensor([-2.5]), torch.tensor([-4.0])),
    ("Política sin preferencia (neutro)",
     torch.tensor([-3.0]), torch.tensor([-3.0]),
     torch.tensor([-3.0]), torch.tensor([-3.0])),
    ("Política ligeramente desalineada",
     torch.tensor([-4.0]), torch.tensor([-2.5]),
     torch.tensor([-2.5]), torch.tensor([-4.0])),
    ("Política completamente invertida",
     torch.tensor([-5.0]), torch.tensor([-1.0]),
     torch.tensor([-3.0]), torch.tensor([-3.0])),
]

results_dpo = []
for name, lpw, lpl, lrw, lrl in scenarios_dpo:
    loss = dpo_loss(lpw, lpl, lrw, lrl).item()
    results_dpo.append((name, loss))
    print(f"{name:<45}  {loss:.4f}")

In [ ]:
# ── Curva DPO loss vs margen de log-ratio ──────────────────────────────────

beta_values = [0.05, 0.1, 0.2, 0.5]
margins_dpo = torch.linspace(-4, 4, 200)

fig, ax = plt.subplots(figsize=(10, 5))
for beta_val in beta_values:
    # π_ref igual para todos → log-ratio_ref = 0 → logits = beta * margin
    losses_dpo = [-F.logsigmoid(beta_val * m).item() for m in margins_dpo]
    ax.plot(margins_dpo.numpy(), losses_dpo, lw=2,
            label=f"β={beta_val}")

ax.axvline(0, color="gray", ls="--", alpha=0.5, label="sin preferencia")
ax.set_xlabel("Margen   log(π_θ(y_w)/π_θ(y_l)) − log(π_ref(y_w)/π_ref(y_l))")
ax.set_ylabel("Loss DPO")
ax.set_title("Loss DPO vs margen de log-ratio para distintos β\n"
             "(margen>0: política prefiere y_w sobre y_l más que la referencia)")
ax.legend(); ax.set_ylim(0, 2); plt.tight_layout(); plt.show()

print("\nInterpretación de β:")
print("  β pequeño (0.05): la política puede alejarse más de π_ref")
print("  β grande  (0.5) : fuerte restricción KL, cambios conservadores")

In [ ]:
# ── Verificación numérica: gradiente DPO ──────────────────────────────────

# Comprobar que el gradiente empuja en la dirección correcta
log_pi_w = torch.tensor([-2.5], requires_grad=True)
log_pi_l = torch.tensor([-2.5], requires_grad=True)
log_ref_w = torch.tensor([-2.5], requires_grad=False)
log_ref_l = torch.tensor([-4.0], requires_grad=False)

loss = dpo_loss(log_pi_w, log_pi_l, log_ref_w, log_ref_l, beta=0.1)
loss.backward()

print("Gradiente de la loss DPO respecto a log_pi_w (debería ser negativo → subir prob de y_w):")
print(f"  ∂L/∂log_π_w = {log_pi_w.grad.item():.6f}  (negativo ✓ → gradient descent sube π_w)")
print(f"  ∂L/∂log_π_l = {log_pi_l.grad.item():.6f}  (positivo ✓ → gradient descent baja π_l)")

# Variantes modernas
print("\n--- Variantes de DPO ---")
variantes = [
    ("IPO",  "Regularización directa en el espacio de logratios; evita over-fitting a pares."),
    ("KTO",  "No necesita pares: solo señal bueno/malo. Más fácil de recolectar."),
    ("ORPO", "Sin modelo de referencia separado: la penalización KL se incorpora inline."),
    ("SimPO","Sin log-normalización por longitud; más estable en secuencias largas."),
]
for name, desc in variantes:
    print(f"  {name:6s}: {desc}")

## 11b. Guía de hiperparámetros LoRA

La elección de hiperparámetros LoRA tiene un impacto enorme en la calidad.
Aquí las reglas empíricas más sólidas de la literatura:

### Tabla de recomendaciones

| Hiperparámetro | Tareas simples | Tareas complejas | Notas |
|----------------|---------------|-----------------|-------|
| `r` | 4–8 | 16–64 | Mayor r = más capacidad pero más parámetros |
| `alpha` | `= 2r` | `= 2r` | La convención α=2r es robusta en la mayoría de casos |
| `lora_dropout` | 0.0–0.05 | 0.05–0.1 | Regularización adicional para datasets pequeños |
| `target_modules` | `q_proj, v_proj` | todas las proyecciones + FFN | Más módulos = más capacidad |
| Learning rate | 1e-4–3e-4 | 5e-5–2e-4 | LoRA permite LR ~10× mayor que FT completo |
| Batch efectivo | 64–128 | 128–512 | Mayor batch → gradientes más estables |

### Reglas de dedo

- **Empezar con r=8, alpha=16**: funciona bien como punto de partida en 90% de los casos.
- **`target_modules`**: Q y V son suficientes para comprensión; añadir K, O, up/down proj
  de las FFN mejora la generación creativa.
- **alpha**: siempre mantener alpha/r constante al variar r (si duplicas r, duplica alpha).
- **Learning rate**: LoRA soporta LR más altas que FT completo porque solo mueve $r(d+k)$
  parámetros; típico 1e-4–3e-4 con cosine decay.

In [ ]:
# ── Análisis: efecto de r sobre capacidad y generalización ──────────────────

# Simulación: modelo toy, dataset de juguete, múltiples rangos
# Métrica: loss de entrenamiento y "test" (generalization gap)

torch.manual_seed(SEED)

# Dataset de juguete con tarea de regresión (análogo a la tarea SFT)
N_TRAIN, N_TEST = 200, 50
X_train = torch.randn(N_TRAIN, 128)
W_true  = torch.randn(64, 128) * 0.1
Y_train = X_train @ W_true.T + 0.01 * torch.randn(N_TRAIN, 64)
X_test  = torch.randn(N_TEST,  128)
Y_test  = X_test  @ W_true.T

rs_hp  = [1, 2, 4, 8, 16, 32, 64]
train_losses_r, test_losses_r = [], []

for r_val in rs_hp:
    base_lin = nn.Linear(128, 64, bias=False)
    for p in base_lin.parameters(): p.data.fill_(0.0)  # inicio neutro
    lora_lin = LoRALinear(base_lin, r=r_val, alpha=float(r_val * 2))

    opt_r = torch.optim.AdamW(
        [p for p in lora_lin.parameters() if p.requires_grad],
        lr=1e-2, weight_decay=1e-3
    )
    for _ in range(300):
        loss_r = F.mse_loss(lora_lin(X_train), Y_train)
        opt_r.zero_grad(); loss_r.backward(); opt_r.step()

    with torch.no_grad():
        tr_l = F.mse_loss(lora_lin(X_train), Y_train).item()
        te_l = F.mse_loss(lora_lin(X_test),  Y_test).item()
    train_losses_r.append(tr_l)
    test_losses_r.append(te_l)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(rs_hp, train_losses_r, "o-", label="Train loss", color="#5b8dd9")
axes[0].plot(rs_hp, test_losses_r,  "s--", label="Test loss",  color="#e07070")
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("Rango r")
axes[0].set_ylabel("MSE"); axes[0].set_title("Loss vs rango r (tarea regresión toy)")
axes[0].legend()

gap = np.array(test_losses_r) - np.array(train_losses_r)
axes[1].bar(np.arange(len(rs_hp)), gap, tick_label=[str(r) for r in rs_hp], color="#70d0a0")
axes[1].set_xlabel("Rango r"); axes[1].set_ylabel("Generalization gap (test - train)")
axes[1].set_title("Overfitting vs rango r\n(gap mayor = más overfitting)")

plt.tight_layout(); plt.show()

print("\nRecomendación general:")
best_r = rs_hp[np.argmin(test_losses_r)]
print(f"  r={best_r} minimiza el test loss en esta tarea toy")
print(f"  En la práctica, r=8 o r=16 es el punto óptimo para la mayoría de tareas SFT")

In [ ]:
# ── Comparación de target_modules ────────────────────────────────────────────

# Cuantificar cómo varía el % de parámetros entrenables según target_modules
module_configs = [
    ("solo atención (qkv)", ("qkv",)),
    ("atención + proj",     ("qkv", "proj")),
    ("atención + FFN",      ("qkv", "proj", "ff1", "ff2")),
]

print(f"{'Config target_modules':<35}  {'Entrenables':>12}  {'%':>6}")
print("-" * 58)
for label, targets in module_configs:
    m_tmp = MiniGPT(vocab_size=len(stoi), d_model=64, n_layers=2, n_heads=4, d_ff=128)
    m_tmp = inject_lora(m_tmp, target_modules=targets, r=8, alpha=16.0)
    tr  = sum(p.numel() for p in m_tmp.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m_tmp.parameters())
    print(f"{label:<35}  {tr:>12,}  {100*tr/tot:>5.2f}%")

print("""
Recomendaciones:
  - Tareas simples (clasificación, formato): q_proj + v_proj es suficiente
  - Generación de calidad / razonamiento   : añadir k_proj, o_proj, up/down en FFN
  - Fine-tuning de dominio técnico          : todas las proyecciones lineales
""")

## 12. Resumen, ejercicios y referencias

### Resumen del notebook

| Sección | Concepto clave | Implementado |
|---------|----------------|-------------|
| 1. Base vs Instruct | 3 etapas de alineamiento | Descripción cualitativa |
| 2. FT completo | 16 B/param con AdamW → impracticable | Calculadora de VRAM |
| 3. Dataset SFT | 30+ ejemplos multi-dominio, estadísticas | Inline en Python |
| 4. Chat templates | ChatML, Alpaca, Llama-3 | `format_chat()` |
| 5. Masking del prompt | `-100` en tokens de instrucción | `build_sft_example()` |
| 6. LoRA desde cero | $\Delta W = \frac{\alpha}{r}BA$, MiniGPT, curva loss | `LoRALinear`, `MiniGPT` |
| 7. PEFT | `get_peft_model`, `merge_and_unload` | try/except HF |
| 8. QLoRA | NF4, double quant, paged optimizers | Comparativa VRAM |
| 9. Before/After | Histogramas A/B/ΔW, norma por capa | Visualizaciones |
| 10. RLHF | Bradley-Terry, PPO, restricción KL | `reward_loss()` |
| 11. DPO | Derivación desde RLHF, verificación numérica | `dpo_loss()` |
| 11b. Hiperparámetros | r, alpha, target_modules, LR | Tabla + análisis |

El código modular está en [`src/llm_lab/finetuning/`](../src/llm_lab/finetuning/).

---

### Ejercicios propuestos

1. **(★)** Modifica `LoRALinear` para que también acepte `dropout` en el camino LoRA
   y verifica que sigue produciendo $\Delta W=0$ al inicio.

2. **(★★)** Implementa `LoRALinear.merge()` que fusione los adaptadores en el peso
   base y verifica con `torch.allclose` que las salidas son idénticas antes y
   después del merge.

3. **(★★)** Entrena el MiniGPT con LoRA solo sobre el dominio `sistemas_potencia`
   del dataset SFT y compara las curvas de loss vs el entrenamiento sobre todos
   los dominios. ¿Qué observas?

4. **(★★★)** Implementa el cálculo de log-probabilidades de una secuencia completa
   $\log P(y|x) = \sum_t \log P_\theta(y_t | y_{<t}, x)$ y entrena la loss DPO
   sobre pares sintéticos (respuestas generadas por el MiniGPT con temperatura
   alta vs baja).

5. **(★★★)** Implementa `IPO` (Identity Preference Optimization): la loss es
   $(\log\frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)} - \frac{1}{2\beta})^2$
   y compara la curva de entrenamiento con DPO.

---

### Referencias

- Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*.
  [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*.
  [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)
- Ouyang et al. (2022). *Training language models to follow instructions (InstructGPT)*.
  [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
- Rafailov et al. (2023). *Direct Preference Optimization*.
  [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
- Azar et al. (2023). *A General Theoretical Paradigm to Understand Learning from Human Feedback (IPO)*.
  [arXiv:2310.12036](https://arxiv.org/abs/2310.12036)
- Zhou et al. (2023). *LIMA: Less Is More for Alignment*.
  [arXiv:2305.11206](https://arxiv.org/abs/2305.11206)
- Loshchilov & Hutter (2019). *Decoupled Weight Decay Regularization (AdamW)*.
  [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
- Karpathy, A. (2023). *nanoGPT*. [GitHub](https://github.com/karpathy/nanoGPT)
- HuggingFace PEFT docs. [docs.huggingface.co/peft](https://huggingface.co/docs/peft)